In [13]:
import torch
import json
import os
import logging
from datetime import datetime
from typing import List, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import warnings
import re
warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [14]:
# Cell 2: Device setup and configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 Using device: {device}")

# Main configuration
CONFIG = {
    "model_id": "meta-llama/Meta-Llama-3-8B-Instruct",  # Use Instruct version
    "use_quantization": True,
    "device": device,
    "max_new_tokens": 300,
    "temperature": 0.3,
    "input_file": "output_entities2.json",
    "output_dir": "lulc_extraction_output",
    "max_sentences": 20,  # Process first 50 sentences for testing
}

# Valid LULC relations
VALID_RELATIONS = [
    "CHANGE_TO", "INCREASES_BY", "DECREASES_BY", "CAUSES", "LOCATED_IN",
    "OCCURS_DURING", "MEASURES", "AFFECTS", "FROM_TO", "ENABLES"
]

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f" Configuration loaded successfully")
print(f" Output directory: {CONFIG['output_dir']}")
print(f" Model: {CONFIG['model_id']}")

🔧 Using device: cuda
 Configuration loaded successfully
 Output directory: lulc_extraction_output
 Model: meta-llama/Meta-Llama-3-8B-Instruct


In [15]:
# Cell 3: Data loading function
def load_preprocessed_data(file_path: str) -> List[Dict]:
    """Load already processed data with sentence and entities keys"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f" Loaded {len(data)} items from {file_path}")
        
        processed_data = []
        for item in data:
            # Clean the sentence text
            sentence = item.get('sentence', '')
            if sentence.startswith("text': '"):
                sentence = sentence[7:]  # Remove "text': '"
            if sentence.endswith("'"):
                sentence = sentence[:-1]  # Remove trailing quote
            
            # Get existing entities (if any)
            entities = item.get('entities', [])
            
            processed_data.append({
                'sentence': sentence,
                'entities': entities,
                'original_data': item
            })
        
        # Print statistics
        total_entities = sum(len(item['entities']) for item in processed_data)
        sentences_with_entities = sum(1 for item in processed_data if item['entities'])
        
        print(f"📊 Processing Statistics:")
        print(f"  - Total sentences: {len(processed_data)}")
        print(f"  - Sentences with entities: {sentences_with_entities}")
        print(f"  - Total entities extracted: {total_entities}")
        if len(processed_data) > 0:
            print(f"  - Average entities per sentence: {total_entities/len(processed_data):.2f}")
        
        return processed_data
        
    except Exception as e:
        logger.error(f"Error loading data: {e}")
        return []

In [16]:
# Cell 4: Model loading function
def load_llama_model(model_id: str, use_quantization: bool = True):
    """Load Llama model with proper configuration"""
    print(f" Loading model: {model_id}")
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        print(f"📝 Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")
        
        # Configure quantization for memory efficiency
        quantization_config = None
        if use_quantization and device.type == "cuda":
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            print("🔧 Using 4-bit quantization")
        
        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto" if device.type == "cuda" else None,
            quantization_config=quantization_config,
            torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
            trust_remote_code=True
        )
        
        print(f"✅ Model loaded successfully on {model.device}")
        return model, tokenizer
        
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise

print(" Model loading function defined")

 Model loading function defined


In [17]:
def build_lulc_extraction_prompt(sentence):
    """Build prompt for joint entity recognition and relation extraction"""
    
    system_message = """You are an expert in Land Use Land Cover (LULC) analysis. Perform joint entity recognition and relation extraction from the given sentence .

**STRICT Entity Types (USE ONLY THESE):**
- CHANGE: Transformation words (increase, decrease, convert, expand, reduce, grow, decline, lost, gained, loss)
- LOC: Location names (countries, cities, specific name of districts, specific name of provinces)
- LULC: Land Use/Land Cover types (forest, cropland, urban area, built-up area, grassland, wetland, water body, bare ground, agricultural land, residential area, degraded land, woody vegetation)
- DATE: Temporal references (years, months, periods, seasons, decades, 1970s, 1980s)
- PERCENT: Percentage values (25%, 10.5%, thirty percent)
- CARDINAL: Numeric values without % (1000, 2.5 million, three, )
- COORDINATES: Geographic coordinates (40.7°N, latitude 23.5)
- SURFACE_UNIT: Area measurements (100 hectares, 50 km², 1000 acres , 500 hectares)
- PROCESS: Environmental processes (deforestation, urbanization, expansion, drought, flooding, desertification, degradation)
- QUANTITY: Other quantities (rate of change, annual loss, total area, large parts)

**ENTITY EXTRACTION RULES:**
1. Extract entities as concise as possible (e.g., "loss" not "observed loss of woody vegetation")
2. List each date separately (e.g., "1970s" and "1980s" as two entities)
3. "desertification" is a PROCESS (the process of becoming desert), not LULC
4. "woody vegetation" or "woody vegetation cover" is LULC

**Relation Types:**
**CRITICAL: USE ONLY THESE EXACT RELATION TYPES:**
CHANGE_TO, INCREASES_BY, DECREASES_BY, CAUSES, LOCATED_IN, OCCURS_DURING, MEASURES, AFFECTS, FROM_TO, ENABLES
**FORBIDDEN RELATIONS** (Never use these):
- HAS, CONTAINS, BELONGS_TO, PART_OF, IS_A, TYPE_OF
- RESULTS_IN, LEADS_TO, TRIGGERS (use CAUSES instead)
- HAPPENS_IN, TAKES_PLACE (use OCCURS_DURING instead)
**CHANGE_TO**: Indicates a direct transformation from one LULC type to another.
-  CORRECT: forest --CHANGE_TO-- cropland (trees cut, land converted to farming)
-  CORRECT: agricultural land --CHANGE_TO-- urban area (farmland developed into city)
-  WRONG: built-up area --CHANGE_TO-- built-up area (same type, just quantity change)
-  WRONG: forest --CHANGE_TO-- forest (same type, just area change)
-  WRONG:the CHANGE_TO must be between 2 diffrence not same lulc 
**INCREASES_BY/DECREASES_BY**: For quantitative changes within same LULC type
-  CORRECT: built-up area --INCREASES_BY-- 12.77% (more built-up area, not transformation)
-  CORRECT: forest --DECREASES_BY-- 25% (less forest area, not transformation)

**Other Relations:**
- CAUSES: Process entity causes a change (deforestation --CAUSES-- forest loss)
- LOCATED_IN: Spatial relationships between one entity and locs (forest --LOCATED_IN-- Brazil)
- OCCURS_DURING: Temporal relationships between one entity and date (change --OCCURS_DURING-- 2018)
- MEASURES: Quantitative relationships between change and percentage or quality  (12.77% --MEASURES-- increase)
- AFFECTS: Impact relationships between process and lulc they must be a real impact (urbanization --AFFECTS-- forest)
- FROM_TO: Value changes between 2 persentage entity or 2 SURFACE_UNIT entity (52.88% --FROM_TO-- 65.5%)
- ENABLES: One process enables another process (deforestation --ENABLES-- urbanization)
**INSTRUCTIONS & REASONING APPROACH:**
1. **THINK STEP-BY-STEP:** Break down your analysis into clear reasoning steps
2. **Entity Identification First:** Scan the sentence for all entity types systematically
3. **Relationship Analysis:** Ask "What happened to what, when, where, and why?"
4. **VALIDATION RULE:** Before adding any relation, check: "Is this EXACTLY one of my 10 approved relation types?"
5. Extract ONLY relations that are explicitly stated 
6.Check that CHANGE_TO only connects different LULC types
7. Use the format: entity_text:ENTITY_LABEL --RELATIONSHIP-- target_entity:TARGET_LABEL
**FEW-SHOT EXAMPLES:**
<|start_header_id|>user<|end_header_id|>
Analyze this sentence: "Between 1995 and 2005, 30% of Brazil's forest area was converted to cropland due to agricultural expansion."
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
THINKING:
Let me break this down systematically:

1. **Entity Identification:**
   - "1995" and "2005" = DATE (temporal references)
   - "30%" = PERCENT (percentage value)
   - "Brazil" = LOC (country name)
   - "forest area" = LULC (land cover type)
   - "cropland" = LULC (land use type)
   - "converted" = CHANGE (transformation word)
   - "agricultural expansion" = PROCESS (environmental process)

2. **Relationship Analysis:**
   - "forest area converted to cropland" → This is a direct transformation between different LULC types, so CHANGE_TO relationship
   - "30% of forest area was converted" → The percentage measures the extent of change, so MEASURES relationship
   - "due to agricultural expansion" → The process caused the conversion, so CAUSES relationship
   - "Brazil's forest area" → Spatial relationship, so LOCATED_IN
   - "Between 1995 and 2005" → Temporal context for the change, so OCCURS_DURING

3. **Rule Verification:**
   - CHANGE_TO: ✓ Different LULC types (forest ≠ cropland)
   - No same-type transformations: ✓
   - Causality makes sense: ✓ Agricultural expansion logically causes land conversion

ENTITIES:
- 1995 | DATE  
- 2005 | DATE  
- 30% | PERCENT  
- forest area | LULC  
- Brazil | LOC  
- cropland | LULC  
- converted | CHANGE  
- agricultural expansion | PROCESS  

RELATIONS:
- forest area:LULC --CHANGE_TO-- cropland:LULC | CONF: HIGH  
- 30%:PERCENT --MEASURES-- converted:CHANGE | CONF: HIGH  
- agricultural expansion:PROCESS --CAUSES-- converted:CHANGE | CONF: HIGH  
- forest area:LULC --LOCATED_IN-- Brazil:LOC | CONF: HIGH  
- converted:CHANGE --OCCURS_DURING-- 1995:DATE | CONF: HIGH  
- converted:CHANGE --OCCURS_DURING-- 2005:DATE | CONF: HIGH  
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Analyze this sentence: "Between 2000 and 2015, desertification in the Sahel region caused a significant loss of grassland."
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
THINKING:
Let me break this down systematically:

1. **Entity Identification:**
   - "2000" and "2015" = DATE (temporal references)
   - "desertification" = PROCESS (environmental process, not LULC - it's the process of becoming desert)
   - "Sahel region" = LOC (specific geographic region)
   - "loss" = CHANGE (transformation word indicating decrease)
   - "grassland" = LULC (land cover type)

2. **Relationship Analysis:**
   - "desertification caused loss" → Process directly caused the change, so CAUSES relationship
   - "loss of grassland" → The change affects the grassland, so AFFECTS relationship  
   - "desertification in the Sahel region" → The process is located in this region, so LOCATED_IN
   - "Between 2000 and 2015" → The loss occurred during this time period, so OCCURS_DURING

3. **Rule Verification:**
   - No CHANGE_TO here because grassland isn't transforming TO another LULC type, it's just being lost
   - CAUSES makes sense: desertification is a process that logically causes loss
   - Temporal logic: loss happens DURING the time period
   - Spatial logic: process happens IN a location

ENTITIES:
- 2000 | DATE  
- 2015 | DATE  
- desertification | PROCESS  
- Sahel region | LOC  
- loss | CHANGE  
- grassland | LULC  

RELATIONS:
- desertification:PROCESS --CAUSES-- loss:CHANGE | CONF: HIGH  
- loss:CHANGE --AFFECTS-- grassland:LULC | CONF: HIGH  
- desertification:PROCESS --LOCATED_IN-- Sahel region:LOC | CONF: HIGH  
- loss:CHANGE --OCCURS_DURING-- 2000:DATE | CONF: HIGH  
- loss:CHANGE --OCCURS_DURING-- 2015:DATE | CONF: HIGH  
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Analyze this sentence: "Urban area expanded from 30 km² in 1990 to 45 km² in 2020 in Shanghai."
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
THINKING:
Let me analyze this step-by-step:

1. **Entity Identification:**
   - "Urban area" = LULC (land use type)
   - "expanded" = CHANGE (transformation word indicating growth)
   - "30 km²" and "45 km²" = SURFACE_UNIT (area measurements)
   - "1990" and "2020" = DATE (temporal references)
   - "Shanghai" = LOC (city name)

2. **Relationship Analysis:**
   - "Urban area in Shanghai" → Spatial relationship, so LOCATED_IN
   - "from 30 km² to 45 km²" → Value change between two surface units, so FROM_TO
   - "30 km² in 1990" → The first measurement occurred during 1990, so OCCURS_DURING
   - "45 km² in 2020" → The second measurement occurred during 2020, so OCCURS_DURING
   - "expanded in 2020" → The expansion happened during this period, so OCCURS_DURING

3. **Rule Verification:**
   - No CHANGE_TO because it's the same LULC type (urban area staying urban area), just expanding
   - Could use INCREASES_BY, but the sentence gives absolute values, not percentages
   - FROM_TO is appropriate for connecting two surface unit measurements
   - This is quantitative change within same LULC type, not transformation to different type

ENTITIES:
- Urban area | LULC  
- expanded | CHANGE  
- 30 km² | SURFACE_UNIT  
- 1990 | DATE  
- 45 km² | SURFACE_UNIT  
- 2020 | DATE  
- Shanghai | LOC  

RELATIONS:
- Urban area:LULC --LOCATED_IN-- Shanghai:LOC | CONF: HIGH  
- 30 km²:SURFACE_UNIT --FROM_TO-- 45 km²:SURFACE_UNIT | CONF: HIGH  
- 30 km²:SURFACE_UNIT --OCCURS_DURING-- 1990:DATE | CONF: HIGH  
- 45 km²:SURFACE_UNIT --OCCURS_DURING-- 2020:DATE | CONF: HIGH  
- expanded:CHANGE --OCCURS_DURING-- 2020:DATE | CONF: HIGH  
<|eot_id|>
**CRITICAL THINKING RULES:**
1. Ask yourself: Is this ACTUALLY a transformation between different land types?
2. Think about the process: What physical change happened to the land?
3. Consider causality: What caused what? Don't create meaningless loops
4. Be precise with measurements: Percentages usually MEASURE changes, not cause them
5. Temporal logic: Changes happen DURING time periods, not TO time periods
6. Spatial logic: Things are LOCATED_IN places, places don't transform to places

INSTRUCTIONS:
1. Extract ONLY relations that are explicitly stated or directly implied in the sentence
2. Use ONLY the entities provided above
3. Each relation must include the entity label in the format: entity_text:ENTITY_LABEL
4. Each relation must follow the format: source_entity:SOURCE_LABEL --RELATIONSHIP-- target_entity:TARGET_LABEL
5. Include confidence level (HIGH/MEDIUM/LOW) for each relation
6. Do not create relations between entities of the same type using CHANGE_TO
**OUTPUT FORMAT:**
ENTITIES:
- entity_text | ENTITY_TYPE

RELATIONS:
- entity:ENTITY_TYPE --RELATIONSHIP-- entity:ENTITY_TYPE | CONF: confidence_level

Extract only what is explicitly mentioned. Be concise and accurate."""

    # Llama 3 format with special tokens
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_message}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Analyze this sentence: "{sentence}"<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
ENTITIES:
-"""
    
    return prompt
print(" Prompt engineering function defined for Llama 3")

 Prompt engineering function defined for Llama 3


In [18]:
def generate_lulc_extraction(sentence, model, tokenizer):
    """Generate entity and relation extraction using Llama 3"""
    prompt = build_lulc_extraction_prompt(sentence)
    
    # Tokenize with proper settings for Llama 3
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,  # Enable truncation for safety
        max_length=4096,    # Llama 3 8B context window
        padding=True,
        return_attention_mask=True
    )
    
    # Move to device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate with Llama 3 optimized settings
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"],
            do_sample=True,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )
    
    # Decode and return
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    return response

print(" Generation function defined")

 Generation function defined


In [19]:
def clean_and_parse_response(response: str, original_sentence: str) -> Dict[str, Any]:
    """Improved parsing that handles both colon and pipe separators"""
    
    result = {
        'entities': [],
        'relations': [],
        'raw_response': response
    }
    
    if not response:
        return result
    
    lines = response.split('\n')
    current_section = "ENTITIES"
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        # Check for section transitions
        if line.upper().startswith("RELATIONS"):
            current_section = "RELATIONS"
            continue
            
        # Parse entities - handles both formats
        if current_section == "ENTITIES":
            # Remove leading dashes/bullets
            clean_line = line.lstrip('-•· ').strip()
            
            # Try colon format: "text:TYPE"
            if ':' in clean_line and not '|' in clean_line:
                parts = clean_line.split(':', 1)
                text = parts[0].strip()
                entity_type = parts[1].strip()
                result['entities'].append({'text': text, 'type': entity_type})
                
            # Try pipe format: "text | TYPE"
            elif '|' in clean_line:
                parts = clean_line.split('|', 1)
                text = parts[0].strip()
                entity_type = parts[1].strip()
                result['entities'].append({'text': text, 'type': entity_type})
                
            # Fallback: Check if line contains a valid entity type
            else:
                valid_types = ['CHANGE', 'LOC', 'LULC', 'DATE', 'PERCENT', 'CARDINAL', 
                              'COORDINATES', 'SURFACE_UNIT', 'PROCESS', 'QUANTITY']
                for etype in valid_types:
                    if etype in line:
                        text = line.replace(etype, '').strip(' :-|')
                        if text:
                            result['entities'].append({'text': text, 'type': etype})
                            
        # Parse relations
        elif current_section == "RELATIONS":
            if "--" in line:
                relation_text = line.lstrip('-•· ').strip()
                result['relations'].append(relation_text)
    
    # Extract missing PERCENT entities
    percentage_pattern = r'\b\d+\.?\d*%\b'
    for relation in result['relations']:
        percentages = re.findall(percentage_pattern, relation)
        for pct in percentages:
            pct_exists = any(entity['text'] == pct for entity in result['entities'])
            if not pct_exists and pct in original_sentence:
                result['entities'].append({'text': pct, 'type': 'PERCENT'})
    
    # Remove duplicates
    seen = set()
    unique_entities = []
    for entity in result['entities']:
        ident = (entity['text'].lower(), entity['type'])
        if ident not in seen:
            seen.add(ident)
            unique_entities.append(entity)
            
    result['entities'] = unique_entities
    return result

# Test with your model output
test_response = """ After the droughts in the 1970s and 1980s | DATE
- loss of woody vegetation cover | CHANGE
- woody vegetation cover | LULC
- Sahel | LOC
- degraded land | LULC

RELATIONS:
- droughts in the 1970s and 1980s:DATE --CAUSES-- loss of woody vegetation cover:CHANGE | HIGH
- loss of woody vegetation cover:CHANGE --AFFECTS-- woody vegetation cover:LULC | HIGH
- Sahel:LOC --LOCATED_IN-- loss of woody vegetation cover:CHANGE | HIGH
- loss of woody vegetation cover:CHANGE --LEADS_TO-- degraded land:LULC | HIGH"""

test_sentence = "After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible and large parts of the Sahel"

parsed = clean_and_parse_response(test_response, test_sentence)
print(" Parsing Results:")
print(f"Entities: {len(parsed['entities'])}")
for e in parsed['entities']:
    print(f" - '{e['text']}' | {e['type']}")
print(f"\nRelations: {len(parsed['relations'])}")
for r in parsed['relations']:
    print(f" - {r}")

 Parsing Results:
Entities: 5
 - 'After the droughts in the 1970s and 1980s' | DATE
 - 'loss of woody vegetation cover' | CHANGE
 - 'woody vegetation cover' | LULC
 - 'Sahel' | LOC
 - 'degraded land' | LULC

Relations: 4
 - droughts in the 1970s and 1980s:DATE --CAUSES-- loss of woody vegetation cover:CHANGE | HIGH
 - loss of woody vegetation cover:CHANGE --AFFECTS-- woody vegetation cover:LULC | HIGH
 - Sahel:LOC --LOCATED_IN-- loss of woody vegetation cover:CHANGE | HIGH
 - loss of woody vegetation cover:CHANGE --LEADS_TO-- degraded land:LULC | HIGH


In [20]:
def process_sentences_batch(sentences: List[Dict], model, tokenizer, max_sentences: int = None) -> List[Dict]:
    """Process multiple sentences and extract LULC information"""
    
    if max_sentences:
        sentences = sentences[:max_sentences]
    
    results = []
    
    print(f" Processing {len(sentences)} sentences...")
    
    for i, item in enumerate(sentences, 1):
        sentence = item['sentence']
        
        if len(sentence) < 10:  # Skip very short sentences
            continue
            
        print(f"📝 Processing {i}/{len(sentences)}: {sentence[:60]}...")
        
        try:
            # Generate extraction
            raw_response = generate_lulc_extraction(sentence, model, tokenizer)
            
            # Parse and clean
            parsed_result = clean_and_parse_response(raw_response, sentence)
            
            # Combine with original data
            result = {
                'sentence': sentence,
                'original_entities': item.get('entities', []),
                'extracted_entities': parsed_result['entities'],
                'extracted_relations': parsed_result['relations'],
                'raw_model_response': parsed_result['raw_response'],
                'processing_timestamp': datetime.now().isoformat()
            }
            
            results.append(result)
            
            # Print progress
            if parsed_result['entities']:
                print(f"    Found {len(parsed_result['entities'])} entities, {len(parsed_result['relations'])} relations")
            else:
                print(f"    No entities found")
                
        except Exception as e:
            logger.error(f"Error processing sentence {i}: {e}")
            result = {
                'sentence': sentence,
                'original_entities': item.get('entities', []),
                'extracted_entities': [],
                'extracted_relations': [],
                'error': str(e),
                'processing_timestamp': datetime.now().isoformat()
            }
            results.append(result)
            continue
            
        # Save intermediate results every 10 sentences
        if i % 10 == 0:
            save_results(results, f"{CONFIG['output_dir']}/intermediate_results_{i}.json")
    
    return results

print("✅ Batch processing function defined")

# Cell 9: Save results function
def save_results(results: List[Dict], output_path: str):
    """Save processing results to JSON file"""
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f" Saved {len(results)} results to {output_path}")
    except Exception as e:
        logger.error(f"Error saving results: {e}")


# Load the model
print("\n🚀 Starting LULC extraction pipeline...")
model, tokenizer = load_llama_model(CONFIG["model_id"], CONFIG["use_quantization"])

# Load input data
print("\n Loading input data...")
input_data = load_preprocessed_data(CONFIG["input_file"])


✅ Batch processing function defined

🚀 Starting LULC extraction pipeline...
 Loading model: meta-llama/Meta-Llama-3-8B-Instruct
📝 Tokenizer loaded. Vocab size: 128000
🔧 Using 4-bit quantization


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded successfully on cuda:0

 Loading input data...
 Loaded 67 items from output_entities2.json
📊 Processing Statistics:
  - Total sentences: 67
  - Sentences with entities: 67
  - Total entities extracted: 334
  - Average entities per sentence: 4.99


In [21]:

# Test with single sentence
if input_data:
    print("\n Testing with single sentence...")
    test_sentence = input_data[2]['sentence']
    print(f"Test sentence: {test_sentence[:150]}...")
    
    # Generate response
    test_response = generate_lulc_extraction(test_sentence, model, tokenizer)
    print(f"\n Raw model response:")
    print(f"'{test_response}'")
    
    # Parse response
    test_parsed = clean_and_parse_response(test_response, test_sentence)
    
    print(f"\n Parsed Results:")
    print(f"   - Entities found: {len(test_parsed['entities'])}")
    for entity in test_parsed['entities']:
        print(f"     • {entity['text']} | {entity['type']}")
    
    print(f"   - Relations found: {len(test_parsed['relations'])}")
    for relation in test_parsed['relations']:
        print(f"     • {relation}")
else:
    print(" No data available for testing")


 Testing with single sentence...
Test sentence: ).',, 'p': 'ref':, ' text': 'From 1996 to 2017, the area covered by cropped fields, C, has increased from 40% to 51.3% over the study site area, and f...

 Raw model response:
' From 1996 to 2017 | DATE
- cropped fields | LULC
- C | LULC
- 40% | PERCENT
- 51.3% | PERCENT
- study site area | SURFACE_UNIT
- 47% | PERCENT
- 64% | PERCENT
- arable lands | LULC
- loss fallows | PROCESS

RELATIONS:
- cropped fields:LULC --LOCATED_IN-- study site area:SURFACE_UNIT | CONF: HIGH
- 40%:PERCENT --FROM_TO-- 51.3%:PERCENT | CONF: HIGH
- 47%:PERCENT --FROM_TO-- 64%:PERCENT | CONF: HIGH
- loss fallows:PROCESS --CAUSES-- cropped fields:LULC | CONF: HIGH
- cropped fields:LULC --AFFECTS-- arable lands:LULC | CONF: HIGH

**REMARKS:**

* The sentence describes a change in the proportion of cropped fields within the study site area and arable lands.
* The process of loss fallows is responsible for the increase in cropped fields.
* The sentence does not menti

In [22]:
if input_data:
    print(f"\n Processing {min(len(input_data), CONFIG['max_sentences'])} sentences...")
    results = process_sentences_batch(
        input_data, 
        model, 
        tokenizer, 
        max_sentences=CONFIG["max_sentences"]
    )
    
    # Save final results
    final_output_path = f"{CONFIG['output_dir']}/llama3_lulc_extraction_results COT 3 shot  .json"
    save_results(results, final_output_path)
    
    # Print summary statistics
    print("\n Final Statistics:")
    print(f"  - Total sentences processed: {len(results)}")
    
    sentences_with_entities = sum(1 for r in results if r['extracted_entities'])
    sentences_with_relations = sum(1 for r in results if r['extracted_relations'])
    total_entities = sum(len(r['extracted_entities']) for r in results)
    total_relations = sum(len(r['extracted_relations']) for r in results)
    
    print(f"  - Sentences with entities: {sentences_with_entities} ({sentences_with_entities/len(results)*100:.1f}%)")
    print(f"  - Sentences with relations: {sentences_with_relations} ({sentences_with_relations/len(results)*100:.1f}%)")
    print(f"  - Total entities extracted: {total_entities}")
    print(f"  - Total relations extracted: {total_relations}")
    print(f"  - Average entities per sentence: {total_entities/len(results):.2f}")
    print(f"  - Average relations per sentence: {total_relations/len(results):.2f}")

# Cell 13: Analyze results
def analyze_extraction_quality(results: List[Dict]):
    """Analyze the quality of extractions"""
    entity_types = {}
    relation_types = {}
    
    for result in results:
        for entity in result['extracted_entities']:
            entity_type = entity['type']
            entity_types[entity_type] = entity_types.get(entity_type, 0) + 1
        
        for relation in result['extracted_relations']:
            # Extract relation type
            if "--" in relation:
                parts = relation.split("--")
                if len(parts) >= 2:
                    rel_type = parts[1].split("--")[0].strip()
                    relation_types[rel_type] = relation_types.get(rel_type, 0) + 1
    
    print("\n Entity Type Distribution:")
    for entity_type, count in sorted(entity_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {entity_type}: {count}")
    
    print("\n Relation Type Distribution:")
    for rel_type, count in sorted(relation_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {rel_type}: {count}")

if 'results' in locals():
    analyze_extraction_quality(results)

print("\n LULC extraction pipeline completed successfully!")


 Processing 20 sentences...
 Processing 20 sentences...
📝 Processing 1/20: 'After the droughts in the 1970s and 1980s, the observed los...
    Found 7 entities, 6 relations
📝 Processing 2/20: 'The forests of West and Central Africa probably originally ...
    Found 4 entities, 3 relations
📝 Processing 3/20: ).',, 'p': 'ref':, ' text': 'From 1996 to 2017, the area cov...
    Found 9 entities, 4 relations
📝 Processing 4/20: in this region has been dominated over the past three decade...
    No entities found
📝 Processing 5/20: land was decreased nearly by half in 2002 compared to its co...
    Found 4 entities, 5 relations
📝 Processing 6/20: About 26 and 31% of the total area of natural vegetation (, ...
    Found 10 entities, 9 relations
📝 Processing 7/20: Assuming the dynamics recorded in the second period, the amo...
    Found 9 entities, 5 relations
📝 Processing 8/20: Conversely the decrease in the herbaceous standing, due to t...
    Found 7 entities, 6 relations
📝 Processing 9/20:

In [23]:
import csv
import re
from typing import List, Dict, Tuple, Optional

# Define approved relations (from your prompt)
APPROVED_RELATIONS = {
    'CHANGE_TO', 'INCREASES_BY', 'DECREASES_BY', 'CAUSES', 
    'LOCATED_IN', 'OCCURS_DURING', 'MEASURES', 'AFFECTS', 
    'FROM_TO', 'ENABLES', 'CHANGES_TO','DECREASED_BY'
}

def parse_relation(relation_str: str) -> Optional[Tuple[str, str, str, str, str, str]]:
    """
    Parse a relation string to extract source, source_type, relationship, target, target_type, and confidence
    
    Expected format: source:SOURCE_TYPE --RELATIONSHIP-- target:TARGET_TYPE | CONF: confidence
    """
    try:
        # Initialize confidence as empty
        confidence = ""
        
        # Extract confidence if present
        if "| CONF:" in relation_str:
            parts = relation_str.split("| CONF:")
            relation_str = parts[0].strip()
            confidence = parts[1].strip()
        elif "|" in relation_str and relation_str.endswith(("HIGH", "MEDIUM", "LOW")):
            # Handle format: ... | HIGH
            parts = relation_str.rsplit("|", 1)
            relation_str = parts[0].strip()
            confidence = parts[1].strip()
        
        # Split by the relationship marker
        if "--" not in relation_str:
            return None
            
        # Find the relationship type (between --)
        parts = relation_str.split("--")
        if len(parts) < 3:
            return None
            
        # Extract components
        source_part = parts[0].strip()
        relationship = parts[1].strip()
        target_part = parts[2].strip()
        
        # Parse source (format: text:TYPE)
        if ":" not in source_part:
            return None
        source_split = source_part.rsplit(":", 1)
        source = source_split[0].strip()
        source_type = source_split[1].strip()
        
        # Parse target (format: text:TYPE)
        if ":" not in target_part:
            return None
        target_split = target_part.rsplit(":", 1)
        target = target_split[0].strip()
        target_type = target_split[1].strip()
        
        return source, source_type, relationship, target, target_type, confidence
        
    except Exception as e:
        print(f"Error parsing relation: {relation_str} - {e}")
        return None

def is_approved_relation(relationship: str) -> bool:
    """
    Check if a relationship type is in the approved list
    """
    return relationship.upper().strip() in APPROVED_RELATIONS

def save_results_to_csv(results: List[Dict], output_path: str, filter_relations: bool = True):
    """
    Save extraction results to CSV with proper relation parsing and filtering
    """
    
    # Prepare CSV data
    csv_rows = []
    sentence_id = 1
    
    # Statistics tracking
    total_relations = 0
    filtered_relations = 0
    approved_relations = 0
    filtered_out_relations = []  # Track what was filtered
    
    print(f"\n📊 Processing results for CSV export...")
    print(f"🔍 Relation filtering: {'ENABLED' if filter_relations else 'DISABLED'}")
    
    for result in results:
        sentence = result['sentence']
        relations = result.get('extracted_relations', [])
        
        if not relations:
            # Add row even if no relations found
            csv_rows.append({
                'sentenceID': sentence_id,
                'sentence': sentence,
                'source': '',
                'source_type': '',
                'relationship': '',
                'target': '',
                'target_type': '',
                'confidence': ''
            })
        else:
            # Process each relation
            valid_relations = 0
            sentence_filtered = 0
            
            for relation in relations:
                total_relations += 1
                parsed = parse_relation(relation)
                
                if parsed:
                    source, source_type, relationship, target, target_type, confidence = parsed
                    
                    # Apply filtering if enabled
                    if filter_relations and not is_approved_relation(relationship):
                        filtered_relations += 1
                        sentence_filtered += 1
                        filtered_out_relations.append({
                            'sentence_id': sentence_id,
                            'relation': relationship,
                            'full_relation': relation
                        })
                        print(f"  FILTERED: {relationship} (not in approved list)")
                        continue
                    
                    # Add approved relation
                    csv_rows.append({
                        'sentenceID': sentence_id,
                        'sentence': sentence,
                        'source': source,
                        'source_type': source_type,
                        'relationship': relationship,
                        'target': target,
                        'target_type': target_type,
                        'confidence': confidence
                    })
                    valid_relations += 1
                    approved_relations += 1
            
            if valid_relations > 0 or sentence_filtered > 0:
                status = f"✅ {valid_relations} approved"
                if sentence_filtered > 0:
                    status += f",  {sentence_filtered} filtered"
                print(f"  Sentence {sentence_id}: {status}")
        
        sentence_id += 1
    
    # Write to CSV
    if csv_rows:
        with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['sentenceID', 'sentence', 'source', 'source_type', 'relationship', 
                         'target', 'target_type', 'confidence']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            
            writer.writeheader()
            writer.writerows(csv_rows)
        
        print(f"\n✅ CSV file saved to: {output_path}")
        print(f"   Total rows: {len(csv_rows)}")
        
        # Print comprehensive statistics
        sentences_with_relations = len(set(row['sentenceID'] for row in csv_rows if row['source']))
        
        print(f"\n📈 Processing Statistics:")
        print(f"   - Total sentences: {sentence_id - 1}")
        print(f"   - Total relations found: {total_relations}")
        print(f"   - Approved relations kept: {approved_relations}")
        print(f"   - Relations filtered out: {filtered_relations}")
        print(f"   - Sentences with valid relations: {sentences_with_relations}")
        
        # Show filtering summary
        if filter_relations and filtered_out_relations:
            print(f"\n Filtered Relations Summary:")
            filtered_types = {}
            for item in filtered_out_relations:
                rel_type = item['relation']
                if rel_type not in filtered_types:
                    filtered_types[rel_type] = 0
                filtered_types[rel_type] += 1
            
            for rel_type, count in sorted(filtered_types.items()):
                print(f"   - {rel_type}: {count} times")
        
        # Show approved relations list
        print(f"\n✅ Approved Relations (only these are kept):")
        for rel in sorted(APPROVED_RELATIONS):
            print(f"   - {rel}")
        
        # Show sample of the CSV content
        print(f"\n📋 Sample CSV content (first 5 approved relations):")
        sample_count = 0
        for row in csv_rows:
            if row['source'] and sample_count < 5:
                print(f"   {row['sentenceID']} | {row['source']}:{row['source_type']} "
                      f"--{row['relationship']}-- {row['target']}:{row['target_type']} "
                      f"| CONF: {row['confidence']}")
                sample_count += 1
    else:
        print("⚠️ No data to save to CSV")

# Execute the CSV export with filtering
if 'results' in locals() and results:
    csv_output_path = f"{CONFIG['output_dir']}/llama3_lulc_relations_3shot.csv"
    save_results_to_csv(results, csv_output_path, filter_relations=True)  # Set to False to disable filtering


📊 Processing results for CSV export...
🔍 Relation filtering: ENABLED
  FILTERED: RESULTS_IN (not in approved list)
  Sentence 1: ✅ 5 approved,  1 filtered
  Sentence 2: ✅ 3 approved
  Sentence 3: ✅ 4 approved
  FILTERED: COMPARED_TO (not in approved list)
  Sentence 5: ✅ 2 approved,  1 filtered
  FILTERED: CONTAINS (not in approved list)
  FILTERED: CONTAINS (not in approved list)
  FILTERED: DEGRADES (not in approved list)
  Sentence 6: ✅ 2 approved,  3 filtered
  Sentence 7: ✅ 5 approved
  FILTERED: DESCRIPTION (not in approved list)
  FILTERED: APPLIES_TO (not in approved list)
  FILTERED: IS_IRRIGATED (not in approved list)
  Sentence 9: ✅ 0 approved,  3 filtered
  FILTERED: DOMINATED (not in approved list)
  FILTERED: CONTINUES_TO_GROW (not in approved list)
  FILTERED: CONTINUES_TO_GROW (not in approved list)
  Sentence 10: ✅ 3 approved,  3 filtered
  FILTERED: HAS_HIGHER_PROPORTION (not in approved list)
  Sentence 11: ✅ 1 approved,  1 filtered
  FILTERED: DOMINANT (not in appr